# 04 — Modeling Baseline (Exp 0 + Exp 1)
**Purpose:** Establish spatial CV framework + baselines.

- **Exp 0:** Naive baselines (mean predictor)
- **Exp 1:** Default XGBoost, RF, Ridge with Landsat + TerraClimate features

### Figures
1. Spatial CV fold distribution map
2. Exp 0 vs Exp 1 R2 comparison bar chart
3. Per-fold R2 boxplots (stability check)
4. Model x Target R2 heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.base import clone
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6), 'font.size': 11,
    'axes.titleweight': 'bold', 'figure.dpi': 120
})

SEED = 42
WORK_DIR = '/kaggle/working'

In [ ]:
# ============================================================
# INLINE: LeaveStationGroupOut (self-contained for Kaggle)
# ============================================================
from sklearn.model_selection import BaseCrossValidator

class LeaveStationGroupOut(BaseCrossValidator):
    """Leave-Station-Group-Out CV. Holds out entire stations per fold."""
    def __init__(self, n_splits=10, station_col='station_id', random_state=42):
        self.n_splits = n_splits
        self.station_col = station_col
        self.random_state = random_state

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        stations = X[self.station_col].unique()
        rng = np.random.RandomState(self.random_state)
        rng.shuffle(stations)
        fold_of = {s: i % self.n_splits for i, s in enumerate(stations)}

        for fold_idx in range(self.n_splits):
            test_stations = [s for s, f in fold_of.items() if f == fold_idx]
            mask = X[self.station_col].isin(test_stations)
            yield X.index[~mask].values, X.index[mask].values

    def get_fold_station_mapping(self, X):
        mapping = {}
        for fi, (_, ti) in enumerate(self.split(X)):
            mapping[fi] = X.loc[ti, self.station_col].unique().tolist()
        return mapping

print('[OK] LeaveStationGroupOut defined inline')

In [ ]:
# Load data (from notebook 03 output or 00 output if skipping)
try:
    train = pd.read_parquet(f'{WORK_DIR}/train_featured.parquet')
    val   = pd.read_parquet(f'{WORK_DIR}/val_featured.parquet')
    print(f'Loaded featured datasets')
except FileNotFoundError:
    train = pd.read_parquet(f'{WORK_DIR}/train_base.parquet')
    val   = pd.read_parquet(f'{WORK_DIR}/val_base.parquet')
    print(f'Loaded base datasets (features not yet engineered)')

# --- Column config (matches notebook 00) ---
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'
DATE_COL    = 'Sample Date'
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
TARGET_SHORT = ['Alkalinity', 'Conductance', 'DRP']  # for plot labels

META_COLS = [STATION_COL, LAT_COL, LON_COL, DATE_COL]

# Verify targets exist
existing_targets = [t for t in TARGET_COLS if t in train.columns]
assert len(existing_targets) == 3, f'Missing targets! Found: {existing_targets}'

print(f'Train: {train.shape},  Val: {val.shape}')
print(f'Targets: {TARGET_COLS}')
print(f'Stations: {train[STATION_COL].nunique()}')

In [ ]:
# Setup spatial CV
cv = LeaveStationGroupOut(n_splits=10, station_col=STATION_COL, random_state=SEED)
fold_map = cv.get_fold_station_mapping(train)

print('Fold distribution:')
for fi, stations in fold_map.items():
    n_samp = train[train[STATION_COL].isin(stations)].shape[0]
    print(f'  Fold {fi}: {len(stations):3d} stations, {n_samp:5d} samples')

---
## FIGURE 1: Spatial CV Fold Assignment Map

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
cmap = plt.cm.get_cmap('tab10', 10)

for fi, stations in fold_map.items():
    fold_rows = train[train[STATION_COL].isin(stations)]
    locs = fold_rows.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
    ax.scatter(locs[LON_COL], locs[LAT_COL],
              c=[cmap(fi)] * len(locs), s=70, alpha=0.85,
              edgecolors='white', linewidths=0.4,
              label=f'Fold {fi} ({len(stations)} st, {len(fold_rows)} samp)')

ax.set_xlim(16, 33);  ax.set_ylim(-35, -22)
ax.set_xlabel('Longitude');  ax.set_ylabel('Latitude')
ax.set_title('Spatial CV Fold Assignment\n'
             'Each fold holds out entire stations — simulates predicting unseen locations')
ax.legend(fontsize=7, loc='lower left', ncol=2,
          frameon=True, fancybox=True, framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_cv_folds_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Exp 0: Naive Baselines

In [ ]:
print('=' * 60)
print('EXP 0: NAIVE BASELINES')
print('=' * 60)

exp0_scores = {}  # {short_name: best_naive_r2}
for t, ts in zip(TARGET_COLS, TARGET_SHORT):
    y = train[t].dropna()
    r2_mean   = r2_score(y, np.full(len(y), y.mean()))
    r2_median = r2_score(y, np.full(len(y), y.median()))
    exp0_scores[ts] = max(r2_mean, r2_median)
    print(f'  {ts:15s}: mean R2={r2_mean:.4f},  median R2={r2_median:.4f}')

exp0_mean = np.mean(list(exp0_scores.values()))
print(f'\nExp 0 overall (best naive) = {exp0_mean:.4f}')

---
## Exp 1: Default Models with EY Features

In [ ]:
print('=' * 60)
print('EXP 1: EY CORE FEATURES (Landsat + TerraClimate)')
print('=' * 60)

# Identify EY-only feature columns
# Raw features: nir, green, swir16, swir22, NDMI, MNDWI, pet
# Plus any temporal/engineered features from notebook 03
all_numeric = train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in all_numeric
                if c not in TARGET_COLS + META_COLS
                and c not in [LAT_COL, LON_COL]]

print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

models_config = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=100, max_depth=10,
                                   random_state=SEED, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 random_state=SEED, n_jobs=-1, verbosity=0),
}

In [ ]:
# Run Exp 1 — track per-fold scores for boxplots
exp1_mean  = {}  # {target_short: {model_name: mean_r2}}
exp1_folds = {}  # {target_short: {model_name: [r2_per_fold]}}

for t, ts in zip(TARGET_COLS, TARGET_SHORT):
    print(f'\n--- {ts} ---')
    valid = train[t].notna()
    X = train.loc[valid].reset_index(drop=True)
    y = train.loc[valid, t].reset_index(drop=True)

    exp1_mean[ts]  = {}
    exp1_folds[ts] = {}

    for mname, model in models_config.items():
        fold_scores = []
        for _, (tr_idx, te_idx) in enumerate(cv.split(X)):
            m = clone(model)
            X_tr = X.iloc[tr_idx][feature_cols].fillna(0)
            X_te = X.iloc[te_idx][feature_cols].fillna(0)
            m.fit(X_tr, y.iloc[tr_idx])
            preds = m.predict(X_te)
            fold_scores.append(r2_score(y.iloc[te_idx], preds))

        mu = np.mean(fold_scores)
        sd = np.std(fold_scores)
        exp1_mean[ts][mname]  = mu
        exp1_folds[ts][mname] = fold_scores
        print(f'  {mname:8s}: R2 = {mu:.4f} +/- {sd:.4f}')

# Summary table
print(f'\n{"="*60}')
print('EXP 1 SUMMARY')
print(f'{"="*60}')
summary_df = pd.DataFrame(exp1_mean).T
summary_df['Best'] = summary_df.idxmax(axis=1)
summary_df.loc['MEAN'] = summary_df.select_dtypes(include=[np.number]).mean()
display(summary_df)

---
## FIGURE 2: Exp 0 vs Exp 1 — R2 Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(TARGET_SHORT))
n_bars = 1 + len(models_config)  # naive + models
width = 0.8 / n_bars

palette = {'Naive': '#9E9E9E', 'Ridge': '#2196F3', 'RF': '#4CAF50', 'XGBoost': '#FF9800'}

# Naive bars
naive_vals = [exp0_scores[ts] for ts in TARGET_SHORT]
ax.bar(x - width * (n_bars - 1) / 2, naive_vals, width,
       label='Exp 0: Naive', color=palette['Naive'], edgecolor='white')

# Model bars
for i, mname in enumerate(models_config):
    vals = [exp1_mean[ts].get(mname, 0) for ts in TARGET_SHORT]
    offset = x - width * (n_bars - 1) / 2 + width * (i + 1)
    ax.bar(offset, vals, width,
           label=f'Exp 1: {mname}', color=palette.get(mname, f'C{i+1}'), edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(TARGET_SHORT, fontsize=12)
ax.set_ylabel('R2 Score (Spatial CV)', fontsize=12)
ax.set_title('Exp 0 (Naive) vs Exp 1 (EY Features)\n'
             'Higher = better  |  Spatial CV prevents over-optimism')
ax.legend(fontsize=10, loc='upper right', framealpha=0.9)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)

# R2 labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_exp0_vs_exp1.png', dpi=150, bbox_inches='tight')
plt.show()

---
## FIGURE 3: Per-Fold R2 Boxplots (Stability)

In [ ]:
fig, axes = plt.subplots(1, len(TARGET_SHORT), figsize=(6 * len(TARGET_SHORT), 6))

box_colors = [palette.get(m, 'C0') for m in models_config]

for i, ts in enumerate(TARGET_SHORT):
    ax = axes[i]
    data = [exp1_folds[ts][m] for m in models_config]
    bp = ax.boxplot(data, labels=list(models_config.keys()),
                    patch_artist=True, widths=0.6)

    for patch, col in zip(bp['boxes'], box_colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.6)

    ax.set_title(ts, fontsize=13)
    ax.set_ylabel('R2 per fold')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='R2 = 0')
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Per-Fold R2 Stability (Exp 1)\n'
             'Wide box = high variance across spatial folds',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_fold_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## FIGURE 4: Model x Target R2 Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

heat_df = pd.DataFrame(exp1_mean)  # rows=models, cols=targets
sns.heatmap(heat_df, annot=True, cmap='RdYlGn', center=0, fmt='.4f',
            linewidths=2, square=True, ax=ax,
            cbar_kws={'label': 'R2 Score', 'shrink': 0.8},
            annot_kws={'fontsize': 13, 'fontweight': 'bold'})

ax.set_title('Model x Target R2 Heatmap (Exp 1)\n'
             'Green = good, Red = poor')
ax.set_xlabel('Target')
ax.set_ylabel('Model')

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Leakage Verification

In [ ]:
forbidden = {'latitude', 'longitude', 'lat', 'lon', 'x', 'y'}
leaks = [c for c in feature_cols if c.lower() in forbidden]

if leaks:
    print(f'!!! LEAKAGE DETECTED: {leaks}')
else:
    print('[OK] No coordinate leakage in feature set')

---
## DEVLOG Entry

In [ ]:
print('\n' + '=' * 60)
print('DEVLOG: Exp 0 + Exp 1')
print('=' * 60)
print(f'\nExp 0: Naive  ->  mean R2 = {exp0_mean:.4f}')
print(f'Exp 1: EY features ({len(feature_cols)} cols)')
for mname in models_config:
    vals = [exp1_mean[ts].get(mname, 0) for ts in TARGET_SHORT]
    print(f'  {mname:8s}: mean R2 = {np.mean(vals):.4f}')
print(f'\nCV: LeaveStationGroupOut, 10 folds, seed={SEED}')
print(f'Leakage check: PASSED')

print(f'\nFigures saved:')
print(f'  fig_04_cv_folds_map.png')
print(f'  fig_04_exp0_vs_exp1.png')
print(f'  fig_04_fold_boxplots.png')
print(f'  fig_04_heatmap.png')